# Notebook 4 — Improved Food Classifier with GAN Augmentation

**Project:** An Improved Computer Vision Model for Food Classification in Smart Refrigerators using GAN-Based Data Augmentation  
**Author:** Premshakthi Sekar | MSc Artificial Intelligence — Northumbria University  

---

## Purpose
This notebook trains the **improved CNN classifier** using the GAN-augmented dataset — real images combined with GAN-generated synthetic images. The results are compared against the baseline model from Notebook 2.

## Key Difference from Baseline
- **Notebook 2 (Baseline):** Trained on original dataset only (~2,000 images)
- **Notebook 4 (Improved):** Trained on original + GAN-generated images (augmented dataset)
- **Epochs reduced:** 200 → 100 (augmented dataset is larger, less training needed)

## Results Summary
| Model | Test Accuracy | Notes |
|-------|--------------|-------|
| Baseline (No GAN) | ~46% | Significant overfitting |
| Improved (With GAN) | ~51% | Marginal improvement, overfitting persists |

> The GAN integration provided a modest improvement in test accuracy. Overfitting remained a challenge, which the dissertation identifies as an area for future work — specifically through model regularisation and more diverse GAN training.

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print('All libraries imported.')

## Step 2: Load GAN-Augmented Dataset

This dataset includes both the original refrigerator images and the GAN-generated synthetic images produced in Notebook 3. The augmented dataset has a larger and more balanced class distribution.

In [ ]:
# Path to GAN-augmented dataset — update to match your local path
# This folder contains both real and GAN-generated images
dataset_path = 'dataset/augmented_images'

images = []
labels = []

classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)
    for filename in os.listdir(class_dir):
        filepath = os.path.join(class_dir, filename)
        if filepath.endswith('.jpg') or filepath.endswith('.png'):
            img = Image.open(filepath).convert('RGB')
            img_array = np.array(img)
            images.append(img_array)
            labels.append(class_name)

print(f'Total images loaded (original + GAN-augmented): {len(images)}')
print(f'Classes: {sorted(set(labels))}')

## Step 3: Compare Class Distribution — Before vs After GAN Augmentation

In [ ]:
from collections import Counter

class_counts = Counter(labels)

plt.figure(figsize=(14, 5))
plt.bar(class_counts.keys(), class_counts.values(), color='#4CAF50', edgecolor='darkgreen', alpha=0.8)
plt.title('Class Distribution — After GAN Augmentation', fontsize=14, fontweight='bold')
plt.xlabel('Food Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('augmented_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution chart saved.')

## Step 4: Encode Labels & Split Dataset

In [ ]:
label_encoder = LabelEncoder()
numerical_labels = label_encoder.fit_transform(labels)

x_train, x_test, y_train, y_test = train_test_split(
    np.array(images), numerical_labels,
    test_size=0.2,
    stratify=numerical_labels,
    random_state=42
)

x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

print(f'Augmented dataset split:')
print(f'  x_train: {x_train.shape}')
print(f'  x_test:  {x_test.shape}')

## Step 5: Build & Train the Improved Model

Same MobileNetV2 architecture as the baseline, but trained on the GAN-augmented dataset. Epochs reduced from 200 to 100 since the augmented dataset is larger.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(640, 640, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(30, activation='softmax')
])

model.compile(
    optimizer=Adam(),
    loss=SparseCategoricalCrossentropy(),
    metrics=[SparseCategoricalAccuracy()]
)

# Train on augmented dataset — 100 epochs (reduced from baseline 200)
history = model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1
)

print('Training complete.')

## Step 6: Evaluate the Improved Model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print('=' * 50)
print('IMPROVED MODEL RESULTS (With GAN Augmentation)')
print('=' * 50)
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')
print(f'Test Loss:     {test_loss:.4f}')
print('=' * 50)
print('\nBaseline (No GAN): ~46%')
print(f'Improvement:       +{(test_acc - 0.46)*100:.1f}%')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['sparse_categorical_accuracy'], label='Train', color='#4CAF50')
ax1.plot(history.history['val_sparse_categorical_accuracy'], label='Validation', color='#FF5722')
ax1.set_title('Improved Model (GAN) — Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'], label='Train', color='#4CAF50')
ax2.plot(history.history['val_loss'], label='Validation', color='#FF5722')
ax2.set_title('Improved Model (GAN) — Loss', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('improved_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix
y_pred_probs   = model.predict(x_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
conf_matrix    = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(12, 10))
sns.heatmap(
    conf_matrix, annot=True, fmt='g',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cmap='Blues'
)
plt.title('Confusion Matrix — Improved Model (GAN Augmented)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

In [ ]:
# Full classification report
print('Classification Report — Improved Model (With GAN):')
print('=' * 60)
print(classification_report(
    y_test, y_pred_classes,
    target_names=label_encoder.classes_
))

---
## Final Summary — Baseline vs GAN-Augmented

| Metric | Baseline (No GAN) | Improved (With GAN) |
|--------|------------------|--------------------|
| Test Accuracy | ~46% | ~51% |
| Overfitting | Significant | Reduced but present |
| Training Epochs | 200 | 100 |
| Dataset Size | Original only | Original + GAN synthetic |

## Key Findings
1. **GAN augmentation provided a modest improvement** in test accuracy (~5% uplift)
2. **Overfitting remained a challenge** — the gap between training and validation accuracy persisted
3. **Class-wise performance varied** — some food categories achieved higher precision/recall than others
4. **GAN image quality** — early-stage GAN images were noisy; further training epochs and architectural improvements could yield better synthetic data quality

## Future Work (from Dissertation)
- Model regularisation techniques (L2, Batch Normalisation)
- More advanced GAN architectures (DCGAN, StyleGAN)
- Real-time inference integration for smart refrigerator systems
- Larger and more diverse training dataset